<a href="https://colab.research.google.com/github/cras-lab/OpenAPI/blob/main/BOK_%EA%B8%B0%EC%A4%80%EA%B8%88%EB%A6%AC%EB%B3%80%ED%99%94.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

필요한 모듈을 임포트 한다.

In [1]:
import requests
import pandas as pd

한국은행 API에 들어갈 매개변수를 설정한다.

In [2]:
# API configuration
START_DATE = "201801"   # 기준금리 읽을 시작 월
END_DATE = "202605"     # 기준금리 읽을 마지막 월
NUM_ITEM = (2026-2018)*12 + 5     # 전체 데이터 개수

외부에서 키를 입력 받는다.

In [ ]:
import getpass
API_KEY = getpass.getpass("ECOS API 키를 입력하시오: ")

ECOS API를 설정한다



In [4]:
url = f"https://ecos.bok.or.kr/api/StatisticSearch/{API_KEY}/json/kr/1/{NUM_ITEM}/722Y001/M/{START_DATE}/{END_DATE}/0101000"

실제 값을 읽어 온다.

In [5]:
# Request
response = requests.get(url)
data = response.json()

StatisticSearch 항목이 없으면 오류를 내고, 그렇지 않으면 row 항목만 추출한다.

In [6]:
# Check and convert to DataFrame
if 'StatisticSearch' not in data:
    import json
    print("❌ Error:\n", json.dumps(data, indent=2, ensure_ascii=False))
    raise SystemExit

rows = data['StatisticSearch']['row']

rows에 들어 있는 값을 json 형식으로 출력해 본다.

In [ ]:
import json
print(json.dumps(rows, indent=2, ensure_ascii=False))

이중, TIME과 DATA_VALUE 만 따로 추출해서 DataFrame으로 만들자.

In [8]:
df = pd.DataFrame(rows) [["TIME", "DATA_VALUE"]]

출력해 보자.

In [ ]:
print(df)
print("Type of DATA_VALUE: ", df["DATA_VALUE"].dtype)

이중 DATA_VALUE 값이 숫자가 아니라 객체이다. 이 부분을 숫자로 변환하자.

In [10]:
df["DATA_VALUE"] = pd.to_numeric(df["DATA_VALUE"])

In [ ]:
print(df["DATA_VALUE"].dtype)


출력해 보자.

In [ ]:
print(df)

<<참고>> 사실 앞의 전과정은 다음의 함수 호출로 한번에 할 수 있다.<BR>
df = (pd.DataFrame(rows).set_index("TIME")[["DATA_VALUE"]])

DataFrame의 내장 함수은 plot()을 이용해 기준금리 변화를 보자.

In [ ]:
df.plot()

Matplotlib를 이용하면 보다 정돈된 그림을 그릴 수 있다.

먼저 문자형태인 TIME 열을 시간형태로 변환한다.

In [ ]:
df['TIME'] = pd.to_datetime(df['TIME'], format='%Y%m')

이제 MATPLOT으로 그래프를 그려본다.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))
plt.plot(df['TIME'], df['DATA_VALUE'], marker='o', linestyle='-')
plt.title("Bank of Korea Base Rate (BOK OpenAPI)", fontsize=14)
plt.xlabel("Period")
plt.ylabel("Policy Interest Rate (%)")
plt.grid(True)
plt.tight_layout()
plt.show()
